In [ ]:
import os, shutil, random, time
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
ORIG_ZIP = "/content/drive/MyDrive/PaddyVarietyBD/PaddyOriginal.zip"
AUG_ZIP  = "/content/drive/MyDrive/PaddyVarietyBD/PaddyAugmented.zip"

EXTRACT_DIR = "/content/paddy_extracted"
BASE_DIR    = "/content/paddy_project"
TRAIN_DIR   = f"{BASE_DIR}/train"
VAL_DIR     = f"{BASE_DIR}/val"

shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
shutil.rmtree(BASE_DIR, ignore_errors=True)

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

In [ ]:
!unzip -q "{ORIG_ZIP}" -d "{EXTRACT_DIR}"
!unzip -q "{AUG_ZIP}"  -d "{EXTRACT_DIR}"

print("Classes found:", sorted(os.listdir(EXTRACT_DIR)))

Classes found: ['BD70', 'Bd30', 'Bd33', 'Bd34', 'Bd49', 'Bd51', 'Bd52', 'Bd56', 'Bd57', 'Bd72', 'Bd75', 'Bd76', 'Bd79', 'Bd85', 'Bd87', 'Bd91', 'Bd93', 'Bd95', 'Binadhan07', 'Binadhan08', 'Binadhan10', 'Binadhan11', 'Binadhan12', 'Binadhan14', 'Binadhan16', 'Binadhan17', 'Binadhan19', 'Binadhan20', 'Binadhan21', 'Binadhan23', 'Binadhan24', 'Binadhan25', 'Binadhan26', 'Br22', 'Br23']


In [ ]:
EXTS = (".jpg",".jpeg",".png",".bmp",".webp")
MAX_TRAIN_PER_CLASS = 400
MAX_VAL_PER_CLASS   = 100

random.seed(42)

for cls in sorted(os.listdir(EXTRACT_DIR)):
    src = os.path.join(EXTRACT_DIR, cls)
    if not os.path.isdir(src):
        continue

    imgs = [f for f in os.listdir(src) if f.lower().endswith(EXTS)]
    random.shuffle(imgs)

    n_train = int(0.8 * len(imgs))
    train_imgs = imgs[:n_train][:MAX_TRAIN_PER_CLASS]
    val_imgs   = imgs[n_train:][:MAX_VAL_PER_CLASS]

    os.makedirs(f"{TRAIN_DIR}/{cls}", exist_ok=True)
    os.makedirs(f"{VAL_DIR}/{cls}", exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(src, img), f"{TRAIN_DIR}/{cls}/{img}")
    for img in val_imgs:
        shutil.copy(os.path.join(src, img), f"{VAL_DIR}/{cls}/{img}")

    print(f"{cls}: {len(train_imgs)} train | {len(val_imgs)} val")

BD70: 400 train | 100 val
Bd30: 400 train | 100 val
Bd33: 400 train | 100 val
Bd34: 400 train | 100 val
Bd49: 400 train | 100 val
Bd51: 400 train | 100 val
Bd52: 400 train | 100 val
Bd56: 400 train | 100 val
Bd57: 400 train | 100 val
Bd72: 400 train | 100 val
Bd75: 400 train | 100 val
Bd76: 400 train | 100 val
Bd79: 400 train | 100 val
Bd85: 400 train | 100 val
Bd87: 400 train | 100 val
Bd91: 400 train | 100 val
Bd93: 400 train | 100 val
Bd95: 400 train | 100 val
Binadhan07: 400 train | 100 val
Binadhan08: 400 train | 100 val
Binadhan10: 400 train | 100 val
Binadhan11: 400 train | 100 val
Binadhan12: 400 train | 100 val
Binadhan14: 400 train | 100 val
Binadhan16: 400 train | 100 val
Binadhan17: 400 train | 100 val
Binadhan19: 400 train | 100 val
Binadhan20: 400 train | 100 val
Binadhan21: 400 train | 100 val
Binadhan23: 400 train | 100 val
Binadhan24: 400 train | 100 val
Binadhan25: 400 train | 100 val
Binadhan26: 400 train | 100 val
Br22: 400 train | 100 val
Br23: 400 train | 100 val


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.ToTensor(),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, train_tf)
val_ds   = datasets.ImageFolder(VAL_DIR, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)

class_names = train_ds.classes
num_classes = len(class_names)

print("Classes:", class_names)

Using device: cpu
Classes: ['BD70', 'Bd30', 'Bd33', 'Bd34', 'Bd49', 'Bd51', 'Bd52', 'Bd56', 'Bd57', 'Bd72', 'Bd75', 'Bd76', 'Bd79', 'Bd85', 'Bd87', 'Bd91', 'Bd93', 'Bd95', 'Binadhan07', 'Binadhan08', 'Binadhan10', 'Binadhan11', 'Binadhan12', 'Binadhan14', 'Binadhan16', 'Binadhan17', 'Binadhan19', 'Binadhan20', 'Binadhan21', 'Binadhan23', 'Binadhan24', 'Binadhan25', 'Binadhan26', 'Br22', 'Br23']


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
    def forward(self, x): return self.net(x)

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, stride, 1, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x): return torch.relu(self.bn(self.pw(self.dw(x))))

class ResidualDownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c1 = ConvBlock(in_ch, out_ch, 2)
        self.c2 = ConvBlock(out_ch, out_ch)
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, 2, bias=False),
            nn.BatchNorm2d(out_ch)
        )
    def forward(self, x):
        return torch.relu(self.c2(self.c1(x)) + self.skip(x))

class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 32),
            nn.MaxPool2d(2)
        )
        self.s1 = ResidualDownBlock(32, 64)
        self.s2 = nn.Sequential(DepthwiseSeparableBlock(64,128,2), ConvBlock(128,128))
        self.s3 = nn.Sequential(DepthwiseSeparableBlock(128,256,2), ConvBlock(256,256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.s1(x)
        x = self.s2(x)
        x = self.s3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

In [ ]:
def train_model(model, name):
    model.to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    tr_acc, va_acc, tr_loss, va_loss = [], [], [], []
    start = time.time()

    for epoch in range(1, EPOCHS+1):
        print(f"\n[{name}] Epoch {epoch}/{EPOCHS}")

        # Train
        model.train()
        correct = total = loss_sum = 0
        for x,y in tqdm(train_loader, desc="Training"):
            x,y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item()
            correct += (out.argmax(1)==y).sum().item()
            total += y.size(0)

        tr_acc.append(correct/total)
        tr_loss.append(loss_sum/len(train_loader))

        # Val
        model.eval()
        correct = total = loss_sum = 0
        y_true, y_pred = [], []

        with torch.no_grad():
            for x,y in tqdm(val_loader, desc="Validating"):
                x,y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out,y)

                loss_sum += loss.item()
                correct += (out.argmax(1)==y).sum().item()
                total += y.size(0)

                y_true.extend(y.cpu().numpy())
                y_pred.extend(out.argmax(1).cpu().numpy())

        va_acc.append(correct/total)
        va_loss.append(loss_sum/len(val_loader))

        print(f"Train Acc={tr_acc[-1]:.3f} | Val Acc={va_acc[-1]:.3f}")

    return tr_acc, va_acc, tr_loss, va_loss, time.time()-start, y_true, y_pred

In [ ]:
# 1. Custom CNN
custom = CustomCNN(num_classes)
c_tr_acc, c_va_acc, c_tr_loss, c_va_loss, c_time, c_y, c_p = train_model(custom, "Custom CNN")

# 2. ResNet Scratch
res_s = models.resnet18(weights=None)
res_s.fc = nn.Linear(res_s.fc.in_features, num_classes)
rs_tr_acc, rs_va_acc, rs_tr_loss, rs_va_loss, rs_time, rs_y, rs_p = train_model(res_s, "ResNet Scratch")


[Custom CNN] Epoch 1/5


Training:   0%|          | 0/438 [00:00<?, ?it/s]

Validating:   0%|          | 0/110 [00:00<?, ?it/s]

Train Acc=0.245 | Val Acc=0.413

[Custom CNN] Epoch 2/5


Training:   0%|          | 0/438 [00:00<?, ?it/s]

...

Train Acc=0.428 | Val Acc=0.335

[ResNet Scratch] Epoch 5/5


Training:   0%|          | 0/438 [00:00<?, ?it/s]

Validating:   0%|          | 0/110 [00:00<?, ?it/s]

Train Acc=0.445 | Val Acc=0.311

[VGG Scratch] Epoch 1/5


Training:   0%|          | 0/438 [00:00<?, ?it/s]

In [ ]:
# 3. VGG Scratch
vgg_s = models.vgg16(weights=None)
vgg_s.classifier[6] = nn.Linear(4096, num_classes)
vs_tr_acc, vs_va_acc, vs_tr_loss, vs_va_loss, vs_time, vs_y, vs_p = train_model(vgg_s, "VGG Scratch")

# 4. ResNet Transfer
res_t = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in res_t.parameters(): p.requires_grad=False
res_t.fc = nn.Linear(res_t.fc.in_features, num_classes)
r_tr_acc, r_va_acc, r_tr_loss, r_va_loss, r_time, r_y, r_p = train_model(res_t, "ResNet Transfer")

# 5. VGG Transfer
vgg_t = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
for p in vgg_t.features.parameters(): p.requires_grad=False
vgg_t.classifier[6] = nn.Linear(4096, num_classes)
v_tr_acc, v_va_acc, v_tr_loss, v_va_loss, v_time, v_y, v_p = train_model(vgg_t, "VGG Transfer")


[VGG Scratch] Epoch 1/5


Training:   0%|          | 0/438 [00:00<?, ?it/s]

Validating:   0%|          | 0/110 [00:00<?, ?it/s]

Train Acc=0.024 | Val Acc=0.029

[VGG Scratch] Epoch 2/5


Training:   0%|          | 0/438 [00:00<?, ?it/s]

In [ ]:
epochs = range(1, EPOCHS+1)

plt.figure(figsize=(12,6))
plt.plot(epochs, c_va_acc, '-o', label="Custom CNN")
plt.plot(epochs, rs_va_acc, '-s', label="ResNet Scratch")
plt.plot(epochs, vs_va_acc, '-^', label="VGG Scratch")
plt.plot(epochs, r_va_acc, '--s', label="ResNet Transfer")
plt.plot(epochs, v_va_acc, '--^', label="VGG Transfer")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
models_cm = [
    ("Custom CNN", c_y, c_p),
    ("ResNet Scratch", rs_y, rs_p),
    ("VGG Scratch", vs_y, vs_p),
    ("ResNet Transfer", r_y, r_p),
    ("VGG Transfer", v_y, v_p),
]

for name,y,p in models_cm:
    plt.figure(figsize=(10,8))
    sns.heatmap(confusion_matrix(y,p), cmap="Blues",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title(name)
    plt.xticks(rotation=90)
    plt.show()

In [ ]:
def metrics(y,p):
    acc = accuracy_score(y,p)
    pr,rc,f1,_ = precision_recall_fscore_support(y,p,average="weighted",zero_division=0)
    return acc,pr,rc,f1

df = pd.DataFrame([
    ["Custom CNN", *metrics(c_y,c_p), c_time],
    ["ResNet Scratch", *metrics(rs_y,rs_p), rs_time],
    ["VGG Scratch", *metrics(vs_y,vs_p), vs_time],
    ["ResNet Transfer", *metrics(r_y,r_p), r_time],
    ["VGG Transfer", *metrics(v_y,v_p), v_time],
], columns=["Model","Accuracy","Precision","Recall","F1","Training Time (sec)"])

df.round(4)